# A parameter sweep with confidence intervals

RobotWin Studio runs every combination of the factors several times, each with its own seed, headless and with the app's own solvers. This notebook varies the air temperature and the chassis mass of a car world and plots how far it gets in two seconds, with each combination's 95 % confidence interval.

Needs: `pip install -e python` from the repository, the headless runner built (`dotnet build tools/RobotWin.Headless -c Release`), a world file (save the **Car** template from the app as `car.rwworld` next to this notebook), and `pandas` / `matplotlib` for the table and the plot.

In [ ]:
from robotwin.experiment import Experiment, Factor, Metric

exp = Experiment(
    world="car.rwworld",
    name="Car: temperature and mass",
    duration=2.0,
    repeats=5,
    seed=1,
    tolerances=True,  # every run draws the parts' values from their tolerance bands
    factors=[
        Factor("environment.AmbientC", [0, 25, 40]),
        Factor("Plaka.mass", start=0.1, stop=0.5, steps=3),
    ],
    metrics=[
        Metric("Plaka.distance", "final", "distance"),
        Metric("Plaka.speed", "mean", "mean speed"),
    ],
)
result = exp.run(progress=lambda done, total: print(f"{done}/{total}", end="\r"))
result

In [ ]:
runs, summary = result.to_pandas()
summary[["environment.AmbientC", "Plaka.mass", "distance mean", "distance ci95_low", "distance ci95_high"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for mass, group in summary.groupby("Plaka.mass"):
    mean = group["distance mean"]
    err = group["distance ci95_high"] - mean
    ax.errorbar(group["environment.AmbientC"], mean, yerr=err, marker="o", capsize=4, label=f"{mass:.2f} kg")
ax.set_xlabel("Air temperature (°C)")
ax.set_ylabel("Distance in 2 s (m)")
ax.legend(title="Chassis mass")
plt.show()

In [ ]:
from IPython.display import Markdown

Markdown(result.report)  # the report RobotWin wrote, with the seeds and settings to repeat it